# Streaming — 01: Structured Streaming Fundamentals

## What you will learn
Spark Structured Streaming treats a **live data stream as an unbounded table**.
You write the same DataFrame/SQL code as for batch, and Spark handles the
incremental execution automatically.

In this notebook you will:
1. Understand the stream as a table mental model
2. Set up a file-based streaming source (the easiest to demo locally)
3. Use different output modes: `append`, `complete`, `update`
4. Apply watermarking to handle late-arriving data
5. Build a stateful aggregation over a sliding window
6. Understand checkpointing and fault tolerance

## The Streaming Mental Model
```
Batch (static):          Streaming (unbounded):
┌─────────────────┐      ┌─────────────────────────────┐
│  Bounded Table  │      │  Unbounded Append-Only Table │
│  id │ value     │      │  id │ value │ timestamp      │
│  1  │  10       │      │  1  │  10   │ 10:00:01        │
│  2  │  20       │      │  2  │  20   │ 10:00:02        │
│  3  │  30       │      │  3  │  30   │ 10:00:05        │
└─────────────────┘      │  4  │  15   │ 10:00:07  ← new │
                         │  ...│  ...  │ ...             │
                         └─────────────────────────────┘

Spark incrementally processes new rows and updates the Result Table.
```


In [1]:
import os, time, datetime, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('streaming-fundamentals')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Gluten: {GLUTEN_ENABLED}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 15:24:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Gluten: False


## Step 1 — Streaming Source Setup

We use a **directory-based streaming source**: Spark monitors a directory
and processes new files as they arrive. This is the simplest way to demo
streaming without needing Kafka.

In production, you would use:
- `readStream.format("kafka")` for Kafka
- `readStream.format("kinesis")` for AWS Kinesis  
- `readStream.format("eventhubs")` for Azure Event Hubs


In [2]:
import pathlib, json, threading, time

# Setup directories
stream_input  = f"{DATA_DIR}/stream_input"
stream_output = f"{DATA_DIR}/stream_output"
stream_ckpt   = f"{DATA_DIR}/stream_checkpoint"

for d in [stream_input, stream_output, stream_ckpt]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# Clean up any previous run
import shutil
for d in [stream_input, stream_output, stream_ckpt]:
    shutil.rmtree(d, ignore_errors=True)
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print("Stream directories ready:")
print(f"  Input     : {stream_input}")
print(f"  Output    : {stream_output}")
print(f"  Checkpoint: {stream_ckpt}")

# Define the schema for incoming events
event_schema = StructType([
    StructField("event_id",   LongType()),
    StructField("user_id",    LongType()),
    StructField("event_type", StringType()),   # click, view, purchase, search
    StructField("product",    StringType()),
    StructField("amount",     DoubleType()),
    StructField("timestamp",  TimestampType()),
])

print("\nEvent schema:")
for f in event_schema.fields:
    print(f"  {f.name:<15} {f.dataType}")

Stream directories ready:
  Input     : /workspace/data/stream_input
  Output    : /workspace/data/stream_output
  Checkpoint: /workspace/data/stream_checkpoint

Event schema:
  event_id        LongType()
  user_id         LongType()
  event_type      StringType()
  product         StringType()
  amount          DoubleType()
  timestamp       TimestampType()


## Step 2 — Create a Streaming DataFrame

`spark.readStream` creates a **streaming DataFrame** — it behaves exactly like
a regular DataFrame for transformations, but is executed incrementally.


In [3]:
# Create streaming reader — watches the input directory for new JSON files
streaming_df = (
    spark.readStream
         .format("json")
         .schema(event_schema)               # ALWAYS define schema for streaming!
         .option("maxFilesPerTrigger", 1)    # process 1 file per batch (for demo)
         .load(stream_input)
)

print("Is streaming:", streaming_df.isStreaming)
print("Schema:")
streaming_df.printSchema()
print()
print("This DataFrame is LAZY — nothing runs until we call writeStream.start()")

Is streaming: True
Schema:
root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)


This DataFrame is LAZY — nothing runs until we call writeStream.start()


## Step 3 — Simple Streaming Query (Append Mode)

The simplest query: filter events and write results as they arrive.

**Output modes:**
- `append` — only new rows added to the result (no aggregation)
- `complete` — entire result table written every trigger (for aggregations)
- `update` — only changed rows written (efficient for aggregations)


In [4]:
# Simple filter and projection — append mode
purchase_stream = (
    streaming_df
    .filter(F.col("event_type") == "purchase")
    .filter(F.col("amount") > 0)
    .select(
        "event_id", "user_id", "product", "amount",
        F.date_format("timestamp", "yyyy-MM-dd HH:mm:ss").alias("event_time")
    )
)

# Start the stream writing to console (for learning — shows results inline)
query1 = (
    purchase_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 5)
    .queryName("purchase_stream")
    .start()
)

print(f"Stream started: {query1.name}")
print(f"Status: {query1.status}")
print()
print("Now let's generate some events to process...")

Stream started: purchase_stream
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

Now let's generate some events to process...


26/04/08 15:24:58 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-00661cef-69ab-42a9-9a53-003bb916365e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/08 15:24:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## Step 4 — Generate Test Events

We simulate an event generator writing JSON files to the input directory.
Each file represents a micro-batch of events arriving from a web application.


In [5]:
import random, json as pyjson
random.seed(42)

EVENT_TYPES = ["click", "view", "purchase", "search", "add_to_cart"]
PRODUCTS    = ["Laptop", "Phone", "Headphones", "Tablet", "Keyboard", "Monitor"]

def generate_batch(batch_id, n_events=20):
    """Generate a batch of random events as a JSON file."""
    now = datetime.datetime.now()
    events = []
    for i in range(n_events):
        ts = now - datetime.timedelta(seconds=random.randint(0, 300))
        events.append({
            "event_id":   batch_id * 1000 + i,
            "user_id":    random.randint(1, 100),
            "event_type": random.choice(EVENT_TYPES),
            "product":    random.choice(PRODUCTS),
            "amount":     round(random.uniform(10, 1000), 2) if random.random() > 0.7 else 0.0,
            "timestamp":  ts.strftime("%Y-%m-%dT%H:%M:%S.%f")
        })
    # Write as JSON lines file
    path = f"{stream_input}/batch_{batch_id:04d}.json"
    with open(path, "w") as f:
        for e in events:
            f.write(pyjson.dumps(e) + "\n")
    return path

# Generate 5 batches
print("Generating event batches...")
for i in range(1, 6):
    path = generate_batch(i)
    print(f"  Batch {i}: {path}")
    time.sleep(2)  # Give stream time to process

# Wait for stream to catch up
time.sleep(3)
print("\nStream progress:")
print(f"  Batches processed: {query1.lastProgress.get('batchId', 'N/A') if query1.lastProgress else 'N/A'}")

Generating event batches...
  Batch 1: /workspace/data/stream_input/batch_0001.json


[Stage 0:>                                                          (0 + 1) / 1]

  Batch 2: /workspace/data/stream_input/batch_0002.json


-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-------+-------+------+-------------------+
|event_id|user_id|product|amount|event_time         |
+--------+-------+-------+------+-------------------+
|1011    |99     |Laptop |867.82|2026-04-08 15:23:02|
|1013    |27     |Monitor|651.56|2026-04-08 15:21:57|
+--------+-------+-------+------+-------------------+



[Stage 1:>                                                          (0 + 1) / 1]

  Batch 3: /workspace/data/stream_input/batch_0003.json


-------------------------------------------
Batch: 1
-------------------------------------------
+--------+-------+--------+------+-------------------+
|event_id|user_id|product |amount|event_time         |
+--------+-------+--------+------+-------------------+
|2006    |93     |Keyboard|512.6 |2026-04-08 15:24:59|
|2015    |52     |Tablet  |456.97|2026-04-08 15:22:21|
+--------+-------+--------+------+-------------------+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-------+-------+------+----------+
|event_id|user_id|product|amount|event_time|
+--------+-------+-------+------+----------+
+--------+-------+-------+------+----------+

  Batch 4: /workspace/data/stream_input/batch_0004.json
-------------------------------------------
Batch: 3
-------------------------------------------
+--------+-------+----------+------+-------------------+
|event_id|user_id|product   |amount|event_time         |
+--------+-------+---------

In [6]:
# Check stream progress
if query1.lastProgress:
    prog = query1.lastProgress
    print("Last batch stats:")
    print(f"  Batch ID         : {prog.get('batchId')}")
    print(f"  Input rows       : {prog.get('numInputRows')}")
    print(f"  Processing rate  : {prog.get('processedRowsPerSecond', 0):.0f} rows/sec")
    print(f"  Trigger duration : {prog.get('triggerExecution', {}).get('totalTime', 0)} ms")

# Stop this stream before starting the next
query1.stop()
print("\nStream stopped.")

Last batch stats:
  Batch ID         : 4
  Input rows       : 2
  Processing rate  : 6 rows/sec
  Trigger duration : 0 ms

Stream stopped.


26/04/08 15:25:11 WARN DAGScheduler: Failed to cancel job group 75ffd160-73e9-43d3-b2b9-148cf6ce4953. Cannot find active jobs for it.
26/04/08 15:25:11 WARN DAGScheduler: Failed to cancel job group 75ffd160-73e9-43d3-b2b9-148cf6ce4953. Cannot find active jobs for it.


## Step 5 — Stateful Aggregation with Watermarking

Real streaming pipelines need aggregations: "how many purchases per user in the last 5 minutes?"

**The late data problem:**
Events arrive out of order. A purchase at 10:05 may arrive at 10:08.
If you aggregate by event time, you need to know when to close a window.

**Watermarking** solves this:
- Set a watermark of 2 minutes → Spark waits 2 minutes after the window end
- Events later than 2 minutes after their window are dropped
- Allows state to be cleaned up and memory to be bounded


In [7]:
# Clean up previous data
shutil.rmtree(stream_input, ignore_errors=True)
shutil.rmtree(stream_ckpt, ignore_errors=True)
pathlib.Path(stream_input).mkdir(parents=True, exist_ok=True)
pathlib.Path(stream_ckpt).mkdir(parents=True, exist_ok=True)

# Re-create streaming reader
streaming_df2 = (
    spark.readStream
         .format("json")
         .schema(event_schema)
         .option("maxFilesPerTrigger", 1)
         .load(stream_input)
)

# Watermarked windowed aggregation
windowed_agg = (
    streaming_df2
    # Add watermark: wait up to 2 minutes for late data
    .withWatermark("timestamp", "2 minutes")
    # Tumbling window: non-overlapping 1-minute intervals
    .groupBy(
        F.window("timestamp", "1 minute"),    # window column
        "event_type"
    )
    .agg(
        F.count("*").alias("event_count"),
        F.sum(F.when(F.col("amount") > 0, F.col("amount")).otherwise(0)).alias("total_amount"),
        F.approx_count_distinct("user_id").alias("unique_users")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "event_type", "event_count", "total_amount", "unique_users"
    )
    # NOTE: orderBy not supported on streaming in update mode
    # Results are sorted by the console sink automatically per batch
)

# Update mode: only write changed windows (more efficient than complete)
query2 = (
    windowed_agg
    .writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", False)
    .option("numRows", 20)
    .option("checkpointLocation", stream_ckpt)
    .queryName("windowed_aggregation")
    .start()
)

print(f"Windowed aggregation stream started: {query2.name}")

Windowed aggregation stream started: windowed_aggregation


26/04/08 15:25:11 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [8]:
# Generate events with realistic timestamps (mostly recent, some late)
def generate_windowed_batch(batch_id, n_events=30):
    now = datetime.datetime.now()
    events = []
    for i in range(n_events):
        # 90% of events are within last 2 minutes, 10% are "late" (3-5 min ago)
        if random.random() < 0.9:
            ts = now - datetime.timedelta(seconds=random.randint(0, 120))
        else:
            ts = now - datetime.timedelta(seconds=random.randint(180, 300))

        amt = round(random.uniform(10, 500), 2) if random.random() > 0.6 else 0.0
        events.append({
            "event_id": batch_id * 1000 + i,
            "user_id":  random.randint(1, 50),
            "event_type": random.choice(EVENT_TYPES),
            "product":  random.choice(PRODUCTS),
            "amount":   amt,
            "timestamp": ts.strftime("%Y-%m-%dT%H:%M:%S.%f")
        })

    path = f"{stream_input}/batch_{batch_id:04d}.json"
    with open(path, "w") as f:
        for e in events:
            f.write(pyjson.dumps(e) + "\n")
    return path

print("Generating windowed event batches...")
for i in range(1, 5):
    path = generate_windowed_batch(i)
    print(f"  Batch {i}: {path}")
    time.sleep(3)

time.sleep(5)
query2.stop()
print("\nWindowed aggregation stream stopped.")

Generating windowed event batches...
  Batch 1: /workspace/data/stream_input/batch_0001.json


26/04/08 15:25:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 6:====>                                                     (2 + 4) / 24]

  Batch 2: /workspace/data/stream_input/batch_0002.json


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------------------+-----------+-----------+------------+------------+
|window_start       |window_end         |event_type |event_count|total_amount|unique_users|
+-------------------+-------------------+-----------+-----------+------------+------------+
|2026-04-08 15:21:00|2026-04-08 15:22:00|view       |1          |0.0         |1           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|add_to_cart|3          |673.47      |3           |
|2026-04-08 15:20:00|2026-04-08 15:21:00|purchase   |1          |0.0         |1           |
|2026-04-08 15:23:00|2026-04-08 15:24:00|purchase   |2          |118.06      |2           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|view       |1          |49.48       |1           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|search     |4          |195.77      |4           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|click      |2          |490.55    

[Stage 7:>                                                          (0 + 1) / 1]

...

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+-------------------+-----------+-----------+------------+------------+
|window_start       |window_end         |event_type |event_count|total_amount|unique_users|
+-------------------+-------------------+-----------+-----------+------------+------------+
|2026-04-08 15:24:00|2026-04-08 15:25:00|add_to_cart|4          |673.47      |4           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|view       |4          |192.23      |3           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|search     |11         |522.26      |10          |
|2026-04-08 15:24:00|2026-04-08 15:25:00|click      |4          |542.29      |3           |
|2026-04-08 15:25:00|2026-04-08 15:26:00|click      |4          |456.25      |3           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|purchase   |8          |731.03      |8           |
|2026-04-08 15:25:00|2026-04-08 15:26:00|add_to_cart|1          |0.0       

-------------------------------------------
Batch: 2
-------------------------------------------
+-------------------+-------------------+-----------+-----------+------------------+------------+
|window_start       |window_end         |event_type |event_count|total_amount      |unique_users|
+-------------------+-------------------+-----------+-----------+------------------+------------+
|2026-04-08 15:24:00|2026-04-08 15:25:00|add_to_cart|9          |2011.03           |9           |
|2026-04-08 15:23:00|2026-04-08 15:24:00|purchase   |5          |330.96000000000004|5           |
|2026-04-08 15:23:00|2026-04-08 15:24:00|click      |1          |150.82            |1           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|search     |16         |1405.67           |14          |
|2026-04-08 15:24:00|2026-04-08 15:25:00|click      |7          |1080.88           |5           |
|2026-04-08 15:25:00|2026-04-08 15:26:00|click      |5          |664.3             |4           |
|2026-04-08 15:24:00|

-------------------------------------------
Batch: 3
-------------------------------------------
+-------------------+-------------------+-----------+-----------+------------------+------------+
|window_start       |window_end         |event_type |event_count|total_amount      |unique_users|
+-------------------+-------------------+-----------+-----------+------------------+------------+
|2026-04-08 15:23:00|2026-04-08 15:24:00|purchase   |9          |535.69            |9           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|view       |5          |534.33            |4           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|click      |10         |1717.2800000000002|7           |
|2026-04-08 15:24:00|2026-04-08 15:25:00|purchase   |16         |1405.9599999999998|12          |
|2026-04-08 15:25:00|2026-04-08 15:26:00|add_to_cart|4          |348.52            |4           |
|2026-04-08 15:25:00|2026-04-08 15:26:00|search     |2          |0.0               |2           |
|2026-04-08 15:25:00|

-------------------------------------------
Batch: 4
-------------------------------------------
+------------+----------+----------+-----------+------------+------------+
|window_start|window_end|event_type|event_count|total_amount|unique_users|
+------------+----------+----------+-----------+------------+------------+
+------------+----------+----------+-----------+------------+------------+


Windowed aggregation stream stopped.


26/04/08 15:25:29 WARN DAGScheduler: Failed to cancel job group abf45d36-f7cf-4288-beda-cb866ef13e28. Cannot find active jobs for it.
26/04/08 15:25:29 WARN DAGScheduler: Failed to cancel job group abf45d36-f7cf-4288-beda-cb866ef13e28. Cannot find active jobs for it.


## Step 6 — Checkpointing and Fault Tolerance

Structured Streaming guarantees **exactly-once processing** through checkpointing.

The checkpoint stores:
1. **Offset log** — which input data has been read
2. **State store** — current aggregation state (for stateful queries)
3. **Commit log** — which batches have been committed to the sink

If the driver crashes and restarts, Spark reads the checkpoint and resumes
from exactly where it left off — no duplicates, no missed data.


In [9]:
# Show checkpoint directory structure
import os as _os

ckpt_path = pathlib.Path(stream_ckpt)
if ckpt_path.exists():
    print("Checkpoint directory structure:")
    for item in sorted(ckpt_path.rglob("*"))[:20]:
        indent = "  " * (len(item.parts) - len(ckpt_path.parts))
        size = f"({item.stat().st_size} bytes)" if item.is_file() else ""
        print(f"{indent}{item.name} {size}")
    print()
    print("Key files:")
    print("  offsets/    — tracks which input rows have been READ")
    print("  commits/    — tracks which batches have been COMMITTED to sink")
    print("  state/      — stores aggregation state for stateful queries")
    print()
    print("On restart, Spark reads these files and resumes from last committed batch.")
else:
    print("Checkpoint directory not found — stream may not have written yet.")

Checkpoint directory structure:
  .metadata.crc (12 bytes)
  commits 
    .0.crc (12 bytes)
    .1.crc (12 bytes)
    .2.crc (12 bytes)
    .3.crc (12 bytes)
    .4.crc (12 bytes)
    0 (41 bytes)
    1 (41 bytes)
    2 (41 bytes)
    3 (41 bytes)
    4 (41 bytes)
  metadata (45 bytes)
  offsets 
    .0.crc (16 bytes)
    .1.crc (16 bytes)
    .2.crc (16 bytes)
    .3.crc (16 bytes)
    .4.crc (16 bytes)
    0 (782 bytes)

Key files:
  offsets/    — tracks which input rows have been READ
  commits/    — tracks which batches have been COMMITTED to sink
  state/      — stores aggregation state for stateful queries

On restart, Spark reads these files and resumes from last committed batch.


## Step 7 — Streaming Query Monitoring

Production streaming jobs need monitoring. Spark provides built-in metrics
through `query.lastProgress` and `query.recentProgress`.

Key metrics to watch:
- `numInputRows` — how much data per batch
- `processedRowsPerSecond` — throughput
- `triggerExecution.totalTime` — batch duration (should be < trigger interval)
- `stateOperators` — state size growth (for stateful queries)


In [10]:
# Demonstrate metrics monitoring
shutil.rmtree(stream_input, ignore_errors=True)
shutil.rmtree(stream_ckpt, ignore_errors=True)
pathlib.Path(stream_input).mkdir(parents=True, exist_ok=True)
pathlib.Path(stream_ckpt).mkdir(parents=True, exist_ok=True)

streaming_df3 = (
    spark.readStream.format("json").schema(event_schema)
         .option("maxFilesPerTrigger", 1).load(stream_input)
)

monitored_query = (
    streaming_df3
    .groupBy("event_type")
    .agg(F.count("*").alias("count"), F.sum("amount").alias("total"))
    .writeStream
    .format("memory")  # in-memory sink — great for testing
    .queryName("metrics_demo")
    .outputMode("complete")
    .option("checkpointLocation", stream_ckpt)
    .start()
)

# Generate events and collect metrics
metrics_log = []
for i in range(1, 6):
    generate_batch(i, 50)
    time.sleep(2)
    if monitored_query.lastProgress:
        p = monitored_query.lastProgress
        metrics_log.append({
            "batch":      p.get("batchId", i),
            "input_rows": p.get("numInputRows", 0),
            "rate":       round(p.get("processedRowsPerSecond", 0), 1),
            "duration_ms":p.get("durationMs", {}).get("triggerExecution", 0)
        })

monitored_query.stop()

# Show in-memory results
print("Current stream state (in-memory sink):")
spark.sql("SELECT * FROM metrics_demo ORDER BY count DESC").show()

print("\nBatch metrics log:")
print(f"{'Batch':>6} {'Input Rows':>12} {'Rate (rows/s)':>15} {'Duration (ms)':>15}")
print("-" * 55)
for m in metrics_log:
    print(f"{m['batch']:>6} {m['input_rows']:>12} {m['rate']:>15} {m['duration_ms']:>15}")

26/04/08 15:25:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/08 15:25:39 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@43d9e021] is aborting.
26/04/08 15:25:39 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@43d9e021] aborted.
26/04/08 15:25:40 WARN DAGScheduler: Failed to cancel job group 1b9dfe8a-36bb-4327-a7b1-4c4c603eb71e. Cannot find active jobs for it.
26/04/08 15:25:40 WARN TaskSetManager: Lost task 9.0 in stage 24.0 (TID 243) (172.18.0.6 executor 0): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 14 cancelled Query metrics_demo [id = 38af78b9-a8da-430e-8826-978aec05e184, runId = 1b9dfe8a-36bb-4327-a7b1-4c4c603eb71e] was stopped SQLSTATE: XXKDA)
26/04/08 15:25

Current stream state (in-memory sink):
+-----------+-----+------------------+
| event_type|count|             total|
+-----------+-----+------------------+
|   purchase|   45|           5825.43|
|       view|   42| 5351.030000000001|
|add_to_cart|   40|           7256.83|
|     search|   38|           3116.35|
|      click|   35|4677.2699999999995|
+-----------+-----+------------------+


Batch metrics log:
 Batch   Input Rows   Rate (rows/s)   Duration (ms)
-------------------------------------------------------
     0           50            13.5            3691
     1           50            27.5            1819
     2           50            28.9            1728
     3           50            28.7            1743


## Summary: Structured Streaming Cheat Sheet

### Source options
```python
# File (local/HDFS/S3)
spark.readStream.format("json").schema(schema).load("path/")

# Kafka
spark.readStream.format("kafka")\
     .option("kafka.bootstrap.servers", "host:9092")\
     .option("subscribe", "topic")\
     .load()

# Rate (testing)
spark.readStream.format("rate").option("rowsPerSecond", 100).load()
```

### Output modes
| Mode | Use when | State cleanup |
|---|---|---|
| `append` | Simple filter/project, no aggregation | Immediate |
| `complete` | Small aggregation results | Never (keep all state) |
| `update` | Large aggregation, only send changes | With watermark |

### Always set a watermark for aggregations
```python
df.withWatermark("timestamp", "10 minutes")
  .groupBy(F.window("timestamp", "5 minutes"), "key")
  .agg(F.count("*"))
```

### Checkpointing is mandatory for production
```python
.writeStream
.option("checkpointLocation", "/path/to/checkpoint")
```

**Next:** `02_streaming_iceberg.ipynb` — streaming writes to Iceberg tables (exactly-once).
